In [1]:
# Cell 1 — Import Libraries
import os
import json
import glob
import yaml
import shutil
from pathlib import Path
from collections import defaultdict

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [2]:
# Cell 2 — Define Paths & Classes
dataset_path = "../datasets"
output_path = "../output/final_dataset"

FINAL_CLASSES = ['Plastic', 'Glass', 'Metal', 'Paper', 'Organic', 'Non-Recyclable']

CLASS_MAPPING = {
    # Plastic
    'PLASTIC': 'Plastic', 'Plastic': 'Plastic',
    'plastic bag': 'Plastic', 'sachet': 'Plastic',
    'straw': 'Plastic', 'Plastic film': 'Plastic',
    'Clear plastic bottle': 'Plastic', 'Other plastic bottle': 'Plastic',
    'Plastic bottle cap': 'Plastic', 'Plastic straw': 'Plastic',
    'Disposable plastic cup': 'Plastic', 'Single-use carrier bag': 'Plastic',
    'Polypropylene bag': 'Plastic', 'Crisp packet': 'Plastic',
    'Spread tub': 'Plastic', 'Tupperware': 'Plastic',
    'Disposable food container': 'Plastic', 'Foam food container': 'Plastic',
    'Foam cup': 'Plastic', 'Other plastic container': 'Plastic',
    'Plastic glooves': 'Plastic', 'Plastic utensils': 'Plastic',
    'Squeezable tube': 'Plastic', 'Plastic lid': 'Plastic',
    'Six pack rings': 'Plastic', 'Styrofoam piece': 'Plastic',
    'Other plastic cup': 'Plastic', 'Other plastic wrapper': 'Plastic',
    'Other plastic': 'Plastic', 'Carded blister pack': 'Plastic',
    'Garbage Bag': 'Plastic', 'Garbage bag': 'Plastic',

    # Glass
    'GLASS': 'Glass', 'Glass': 'Glass',
    'Glass bottle': 'Glass', 'Broken glass': 'Glass',
    'Glass jar': 'Glass', 'Glass cup': 'Glass',

    # Metal
    'METAL': 'Metal', 'Metal': 'Metal',
    'Drink can': 'Metal', 'Food Can': 'Metal',
    'Aerosol': 'Metal', 'Scrap metal': 'Metal',
    'Metal bottle cap': 'Metal', 'Metal lid': 'Metal',
    'Pop tab': 'Metal', 'Aluminium foil': 'Metal',
    'Aluminium blister pack': 'Metal',

    # Paper
    'PAPER': 'Paper', 'Paper': 'Paper',
    'CARDBOARD': 'Paper', 'Paper cup': 'Paper',
    'Paper bag': 'Paper', 'Paper straw': 'Paper',
    'Normal paper': 'Paper', 'Magazine paper': 'Paper',
    'Tissues': 'Paper', 'Wrapping paper': 'Paper',
    'Corrugated carton': 'Paper', 'Other carton': 'Paper',
    'Egg carton': 'Paper', 'Drink carton': 'Paper',
    'Meal carton': 'Paper', 'Pizza box': 'Paper',
    'Toilet tube': 'Paper', 'Plastified paper bag': 'Paper',

    # Organic
    'Organic': 'Organic', 'Animal Waste': 'Organic',
    'Food waste': 'Organic',

    # Non-Recyclable
    'Cigarette': 'Non-Recyclable', 'Rope & strings': 'Non-Recyclable',
    'Shoe': 'Non-Recyclable', 'Battery': 'Non-Recyclable',
}

REMOVE_CLASSES = [
    'BIODEGRADABLE', 'litter', 'pile', 'Pile', 'waste',
    'Unlabeled litter', 'trash bin', 'Trash bin',
    'face mask', 'Face mask', 'rope',
    'Construction Waste', 'bottle', 'cup'
]

print("✅ Paths & Classes defined successfully")
print(f"📁 Output path: {os.path.abspath(output_path)}")

✅ Paths & Classes defined successfully
📁 Output path: d:\Master_Waste_Dataset\output\final_dataset


In [3]:
# Cell 3 — Create Output Folder Structure
splits = ['train', 'val', 'test']

for split in splits:
    for folder in ['images', 'labels']:
        path = os.path.join(output_path, split, folder)
        os.makedirs(path, exist_ok=True)

print("✅ Output folder structure created:")
print(f"""
final_dataset/
├── train/
│   ├── images/
│   └── labels/
├── val/
│   ├── images/
│   └── labels/
└── test/
    ├── images/
    └── labels/
""")

✅ Output folder structure created:

final_dataset/
├── train/
│   ├── images/
│   └── labels/
├── val/
│   ├── images/
│   └── labels/
└── test/
    ├── images/
    └── labels/



In [4]:
# Cell 4 — Process YOLO Datasets (Garbage_Detection + Pakistan_Garbage)
import random
random.seed(42)

def process_yolo_dataset(dataset_name):
    dataset_dir = os.path.join(dataset_path, dataset_name)
    
    yaml_files = glob.glob(f"{dataset_dir}/**/*.yaml", recursive=True)
    if not yaml_files:
        print(f"❌ مش لاقي yaml في {dataset_name}")
        return
    
    with open(yaml_files[0], 'r') as f:
        yaml_data = yaml.safe_load(f)
    class_names = yaml_data.get('names', [])
    if isinstance(class_names, dict):
        class_names = [class_names[i] for i in sorted(class_names.keys())]
    
    # جمع كل الصور
    image_files = glob.glob(f"{dataset_dir}/**/*.jpg", recursive=True) + \
                  glob.glob(f"{dataset_dir}/**/*.png", recursive=True) + \
                  glob.glob(f"{dataset_dir}/**/*.jpeg", recursive=True)
    
    copied = 0
    skipped = 0
    
    for img_path in image_files:
        img_stem = Path(img_path).stem
        img_ext = Path(img_path).suffix
        
        # دور على الـ label file
        label_path = None
        for txt in glob.glob(f"{dataset_dir}/**/{img_stem}.txt", recursive=True):
            label_path = txt
            break
        
        if not label_path or not os.path.exists(label_path):
            skipped += 1
            continue
        
        # اقرأ الـ labels وحول الـ classes
        new_lines = []
        with open(label_path, 'r') as f:
            lines = f.readlines()
        
        for line in lines:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            if len(parts) < 5:
                continue
            try:
                class_id = int(parts[0])
            except ValueError:
                continue
            if class_id >= len(class_names):
                continue
            
            class_name = class_names[class_id]
            
            if class_name in REMOVE_CLASSES:
                continue
            elif class_name in CLASS_MAPPING:
                new_class_id = FINAL_CLASSES.index(CLASS_MAPPING[class_name])
                new_lines.append(f"{new_class_id} {' '.join(parts[1:])}\n")
        
        if not new_lines:
            skipped += 1
            continue
        
        # حدد الـ split
        r = random.random()
        if r < 0.7:
            split = 'train'
        elif r < 0.85:
            split = 'val'
        else:
            split = 'test'
        
        # اعمل اسم جديد للملف عشان ميحصلش تعارض
        new_name = f"{dataset_name}_{img_stem}"
        
        # انسخ الصورة
        dst_img = os.path.join(output_path, split, 'images', f"{new_name}{img_ext}")
        shutil.copy2(img_path, dst_img)
        
        # اكتب الـ label الجديد
        dst_label = os.path.join(output_path, split, 'labels', f"{new_name}.txt")
        with open(dst_label, 'w') as f:
            f.writelines(new_lines)
        
        copied += 1
    
    print(f"✅ {dataset_name}: {copied} صورة اتنسخت، {skipped} اتسكبت")

# شغّل على الـ YOLO datasets
process_yolo_dataset("Garbage_Detection")
process_yolo_dataset("Pakistan_Garbage")

✅ Garbage_Detection: 8387 صورة اتنسخت، 2077 اتسكبت
✅ Pakistan_Garbage: 9643 صورة اتنسخت، 2624 اتسكبت


In [5]:
# Cell 5 — Process COCO Datasets (TACO + pLitterStreet)

def process_coco_dataset(dataset_name):
    dataset_dir = os.path.join(dataset_path, dataset_name)
    
    json_files = glob.glob(f"{dataset_dir}/**/*.json", recursive=True)
    if not json_files:
        print(f"❌ مش لاقي json في {dataset_name}")
        return
    
    copied = 0
    skipped = 0
    
    for json_file in json_files:
        with open(json_file, 'r') as f:
            data = json.load(f)
        
        if 'categories' not in data or 'annotations' not in data or 'images' not in data:
            continue
        
        categories = {cat['id']: cat['name'] for cat in data['categories']}
        
        # جمع الـ annotations لكل صورة
        img_annotations = defaultdict(list)
        for ann in data['annotations']:
            class_name = categories.get(ann['category_id'], '')
            if class_name in REMOVE_CLASSES:
                continue
            elif class_name in CLASS_MAPPING:
                new_class_id = FINAL_CLASSES.index(CLASS_MAPPING[class_name])
                bbox = ann['bbox']  # [x, y, width, height] COCO format
                img_annotations[ann['image_id']].append({
                    'class_id': new_class_id,
                    'bbox': bbox,
                    'img_w': None,
                    'img_h': None
                })
        
        # جمع معلومات الصور
        images_info = {img['id']: img for img in data['images']}
        
        for img_id, anns in img_annotations.items():
            if img_id not in images_info:
                skipped += 1
                continue
            
            img_info = images_info[img_id]
            img_w = img_info['width']
            img_h = img_info['height']
            img_file_name = img_info['file_name']
            
            # دور على الصورة
            img_path = None
            for ext in ['jpg', 'jpeg', 'png']:
                possible = glob.glob(f"{dataset_dir}/**/{Path(img_file_name).name}", recursive=True)
                if possible:
                    img_path = possible[0]
                    break
            
            if not img_path or not os.path.exists(img_path):
                skipped += 1
                continue
            
            # حول الـ bbox من COCO لـ YOLO
            yolo_lines = []
            for ann in anns:
                x, y, w, h = ann['bbox']
                x_center = (x + w / 2) / img_w
                y_center = (y + h / 2) / img_h
                w_norm = w / img_w
                h_norm = h / img_h
                yolo_lines.append(f"{ann['class_id']} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n")
            
            if not yolo_lines:
                skipped += 1
                continue
            
            # حدد الـ split
            r = random.random()
            if r < 0.7:
                split = 'train'
            elif r < 0.85:
                split = 'val'
            else:
                split = 'test'
            
            img_stem = Path(img_path).stem
            img_ext = Path(img_path).suffix
            new_name = f"{dataset_name}_{img_stem}"
            
            # انسخ الصورة
            dst_img = os.path.join(output_path, split, 'images', f"{new_name}{img_ext}")
            shutil.copy2(img_path, dst_img)
            
            # اكتب الـ label
            dst_label = os.path.join(output_path, split, 'labels', f"{new_name}.txt")
            with open(dst_label, 'w') as f:
                f.writelines(yolo_lines)
            
            copied += 1
    
    print(f"✅ {dataset_name}: {copied} صورة اتنسخت، {skipped} اتسكبت")

# شغّل على الـ COCO datasets
process_coco_dataset("TACO")
process_coco_dataset("pLitterStreet")

✅ TACO: 2962 صورة اتنسخت، 0 اتسكبت
✅ pLitterStreet: 26875 صورة اتنسخت، 3 اتسكبت


In [6]:
# Cell 6 — Verify Final Dataset
print("📊 إحصائيات الـ Final Dataset:\n")

total_images = 0
total_labels = 0

for split in ['train', 'val', 'test']:
    images = glob.glob(f"{output_path}/{split}/images/*")
    labels = glob.glob(f"{output_path}/{split}/labels/*")
    total_images += len(images)
    total_labels += len(labels)
    print(f"  {split}:")
    print(f"    صور:   {len(images)}")
    print(f"    labels: {len(labels)}")

print(f"\n  📊 الإجمالي:")
print(f"    صور:   {total_images}")
print(f"    labels: {total_labels}")

# تحقق من الـ class distribution
print("\n📊 توزيع الـ Classes في الـ Final Dataset:\n")
class_counts = defaultdict(int)

for label_file in glob.glob(f"{output_path}/**/*.txt", recursive=True):
    with open(label_file, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            class_id = int(line.split()[0])
            class_counts[FINAL_CLASSES[class_id]] += 1

for cls in FINAL_CLASSES:
    count = class_counts[cls]
    bar = "█" * (count // 5000)
    print(f"  {cls:<20} {count:>8} {bar}")

📊 إحصائيات الـ Final Dataset:

  train:
    صور:   24216
    labels: 24216
  val:
    صور:   6487
    labels: 6487
  test:
    صور:   6405
    labels: 6405

  📊 الإجمالي:
    صور:   37108
    labels: 37108

📊 توزيع الـ Classes في الـ Final Dataset:

  Plastic                106865 █████████████████████
  Glass                   11083 ██
  Metal                    9335 █
  Paper                   12007 ██
  Organic                  6125 █
  Non-Recyclable            228 


In [7]:
# Cell 7 — Check Non-Recyclable Sources
print("🔍 تحقق من مصادر الـ Non-Recyclable:\n")

non_rec_classes = ['Cigarette', 'Rope & strings', 'Shoe', 'Battery']

for dataset_name, fmt in [("TACO", "coco"), ("Pakistan_Garbage", "yolo")]:
    dataset_dir = os.path.join(dataset_path, dataset_name)
    print(f"📁 {dataset_name}:")
    
    if fmt == "coco":
        json_files = glob.glob(f"{dataset_dir}/**/*.json", recursive=True)
        for json_file in json_files:
            with open(json_file, 'r') as f:
                data = json.load(f)
            if 'categories' not in data:
                continue
            categories = {cat['id']: cat['name'] for cat in data['categories']}
            counts = defaultdict(int)
            for ann in data['annotations']:
                name = categories.get(ann['category_id'], '')
                if name in non_rec_classes:
                    counts[name] += 1
            for name, count in counts.items():
                print(f"  - {name}: {count}")
    
    elif fmt == "yolo":
        yaml_files = glob.glob(f"{dataset_dir}/**/*.yaml", recursive=True)
        with open(yaml_files[0], 'r') as f:
            yaml_data = yaml.safe_load(f)
        class_names = yaml_data.get('names', [])
        if isinstance(class_names, dict):
            class_names = [class_names[i] for i in sorted(class_names.keys())]
        
        counts = defaultdict(int)
        for txt in glob.glob(f"{dataset_dir}/**/*.txt", recursive=True):
            with open(txt, 'r') as f:
                for line in f:
                    line = line.strip()
                    if not line or line.startswith('#'):
                        continue
                    parts = line.split()
                    if len(parts) < 5:
                        continue
                    try:
                        cid = int(parts[0])
                    except:
                        continue
                    if cid < len(class_names) and class_names[cid] in non_rec_classes:
                        counts[class_names[cid]] += 1
        for name, count in counts.items():
            print(f"  - {name}: {count}")
    print()

🔍 تحقق من مصادر الـ Non-Recyclable:

📁 TACO:
  - Shoe: 7
  - Battery: 2
  - Rope & strings: 29
  - Cigarette: 667
  - Shoe: 2
  - Battery: 1
  - Rope & strings: 1
  - Cigarette: 27
  - Cigarette: 83
  - Shoe: 1
  - Rope & strings: 4
  - Cigarette: 14
  - Rope & strings: 2
  - Shoe: 1
  - Cigarette: 67
  - Rope & strings: 2
  - Cigarette: 10
  - Rope & strings: 2
  - Battery: 1
  - Rope & strings: 2
  - Cigarette: 6
  - Cigarette: 32
  - Shoe: 2
  - Cigarette: 95
  - Shoe: 1
  - Cigarette: 36
  - Rope & strings: 7
  - Cigarette: 28
  - Rope & strings: 2
  - Cigarette: 46
  - Rope & strings: 1
  - Cigarette: 55
  - Rope & strings: 2
  - Cigarette: 38
  - Rope & strings: 1
  - Cigarette: 71
  - Cigarette: 59
  - Rope & strings: 3

📁 Pakistan_Garbage:



In [8]:
# Cell 8 — Aggregate Non-Recyclable Total
print("📊 إجمالي الـ Non-Recyclable في كل Dataset:\n")

non_rec_classes = ['Cigarette', 'Rope & strings', 'Shoe', 'Battery']

for dataset_name, fmt in [("TACO", "coco"), ("Pakistan_Garbage", "yolo")]:
    dataset_dir = os.path.join(dataset_path, dataset_name)
    total_counts = defaultdict(int)
    
    if fmt == "coco":
        json_files = glob.glob(f"{dataset_dir}/**/*.json", recursive=True)
        for json_file in json_files:
            with open(json_file, 'r') as f:
                data = json.load(f)
            if 'categories' not in data:
                continue
            categories = {cat['id']: cat['name'] for cat in data['categories']}
            for ann in data['annotations']:
                name = categories.get(ann['category_id'], '')
                if name in non_rec_classes:
                    total_counts[name] += 1
    
    elif fmt == "yolo":
        yaml_files = glob.glob(f"{dataset_dir}/**/*.yaml", recursive=True)
        with open(yaml_files[0], 'r') as f:
            yaml_data = yaml.safe_load(f)
        class_names = yaml_data.get('names', [])
        if isinstance(class_names, dict):
            class_names = [class_names[i] for i in sorted(class_names.keys())]
        for txt in glob.glob(f"{dataset_dir}/**/*.txt", recursive=True):
            with open(txt, 'r') as f:
                for line in f:
                    line = line.strip()
                    if not line or line.startswith('#'):
                        continue
                    parts = line.split()
                    if len(parts) < 5:
                        continue
                    try:
                        cid = int(parts[0])
                    except:
                        continue
                    if cid < len(class_names) and class_names[cid] in non_rec_classes:
                        total_counts[class_names[cid]] += 1

    print(f"📁 {dataset_name}:")
    grand_total = 0
    for name, count in sorted(total_counts.items(), key=lambda x: -x[1]):
        print(f"  - {name}: {count}")
        grand_total += count
    print(f"  الإجمالي: {grand_total}\n")

📊 إجمالي الـ Non-Recyclable في كل Dataset:

📁 TACO:
  - Cigarette: 1334
  - Rope & strings: 58
  - Shoe: 14
  - Battery: 4
  الإجمالي: 1410

📁 Pakistan_Garbage:
  الإجمالي: 0



In [9]:
# Cell 9 — Check TACO Image Lookup Problem
print("🔍 فحص مشكلة الـ Image Lookup في TACO:\n")

dataset_dir = os.path.join(dataset_path, "TACO")
json_files = glob.glob(f"{dataset_dir}/**/*.json", recursive=True)

total_images = 0
found_images = 0
not_found = 0

for json_file in json_files:
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    if 'images' not in data:
        continue
    
    for img in data['images']:
        total_images += 1
        img_name = Path(img['file_name']).name
        possible = glob.glob(f"{dataset_dir}/**/{img_name}", recursive=True)
        if possible:
            found_images += 1
        else:
            not_found += 1
            if not_found <= 5:
                print(f"  ❌ مش لاقيها: {img['file_name']}")

print(f"\n  إجمالي الصور في الـ JSON: {total_images}")
print(f"  ✅ اتلاقت: {found_images}")
print(f"  ❌ مش اتلاقتش: {not_found}")

🔍 فحص مشكلة الـ Image Lookup في TACO:


  إجمالي الصور في الـ JSON: 3000
  ✅ اتلاقت: 3000
  ❌ مش اتلاقتش: 0


In [10]:
# Cell 10 — Check Duplicate Images Across JSON Files in TACO
print("🔍 فحص التكرار في صور TACO:\n")

dataset_dir = os.path.join(dataset_path, "TACO")
json_files = glob.glob(f"{dataset_dir}/**/*.json", recursive=True)

image_to_jsons = defaultdict(list)

for json_file in json_files:
    with open(json_file, 'r') as f:
        data = json.load(f)
    if 'images' not in data:
        continue
    for img in data['images']:
        img_name = Path(img['file_name']).name
        image_to_jsons[img_name].append(Path(json_file).name)

duplicates = {k: v for k, v in image_to_jsons.items() if len(v) > 1}

print(f"  إجمالي الصور الفريدة: {len(image_to_jsons)}")
print(f"  صور موجودة في أكتر من JSON: {len(duplicates)}")

if duplicates:
    print("\n  أمثلة:")
    for img, jsons in list(duplicates.items())[:5]:
        print(f"  - {img} → {jsons}")

🔍 فحص التكرار في صور TACO:

  إجمالي الصور الفريدة: 339
  صور موجودة في أكتر من JSON: 339

  أمثلة:
  - 000006.jpg → ['annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annotations.json']
  - 000008.jpg → ['annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annotations.json']
  - 000010.jpg → ['annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annotations.json', 'annot

In [11]:
# Cell 11 — Fix TACO Processing (folder by folder)
print("🔍 شكل فولدرات TACO:\n")

dataset_dir = os.path.join(dataset_path, "TACO")

for item in sorted(os.listdir(dataset_dir)):
    item_path = os.path.join(dataset_dir, item)
    if os.path.isdir(item_path):
        json_exists = os.path.exists(os.path.join(item_path, "annotations.json"))
        images = glob.glob(f"{item_path}/*.jpg") + glob.glob(f"{item_path}/*.png")
        print(f"  📁 {item} → {len(images)} صورة, annotations.json: {json_exists}")

🔍 شكل فولدرات TACO:

  📁 TACO → 0 صورة, annotations.json: False
  📁 __MACOSX → 0 صورة, annotations.json: False


In [12]:
# Cell 12 — Explore TACO Structure
print("🔍 استكشاف TACO بالتفصيل:\n")

dataset_dir = os.path.join(dataset_path, "TACO")

for root, dirs, files in os.walk(dataset_dir):
    # بس أول 3 مستويات
    level = root.replace(dataset_dir, '').count(os.sep)
    if level > 2:
        continue
    indent = '  ' * level
    print(f"{indent}📁 {os.path.basename(root)}/")
    for file in files[:5]:  # أول 5 ملفات بس
        print(f"{indent}  📄 {file}")
    if len(files) > 5:
        print(f"{indent}  ... و {len(files)-5} ملف تاني")

🔍 استكشاف TACO بالتفصيل:

📁 TACO/
  📁 TACO/
    📄 .DS_Store
    📄 .gitignore
    📄 demo.ipynb
    📄 download.py
    📄 LICENSE
    ... و 2 ملف تاني
    📁 .git/
      📄 .DS_Store
      📄 COMMIT_EDITMSG
      📄 config
      📄 description
      📄 FETCH_HEAD
      ... و 4 ملف تاني
    📁 .ipynb_checkpoints/
      📄 demo-checkpoint.ipynb
    📁 data/
      📄 .DS_Store
      📄 annotations.json
    📁 detector/
      📄 .DS_Store
      📄 config.py
      📄 dataset.py
      📄 detector.py
      📄 inspect_data.ipynb
      ... و 4 ملف تاني
  📁 __MACOSX/
    📁 TACO/
      📄 ._.DS_Store


In [13]:
# Cell 13 — Find TACO Images Location
print("🔍 دور على الصور في TACO:\n")

dataset_dir = os.path.join(dataset_path, "TACO")

# اقرأ الـ annotations
ann_file = os.path.join(dataset_dir, "TACO", "data", "annotations.json")
with open(ann_file, 'r') as f:
    data = json.load(f)

print(f"  عدد الصور في الـ JSON: {len(data['images'])}")
print(f"  عدد الـ Annotations: {len(data['annotations'])}")
print(f"\n  أمثلة على file_name:")
for img in data['images'][:5]:
    print(f"  - {img['file_name']}")

# دور على الصور فعلياً
print(f"\n  جاري البحث عن الصور...")
all_images = glob.glob(f"{dataset_dir}/**/*.jpg", recursive=True) + \
             glob.glob(f"{dataset_dir}/**/*.png", recursive=True)
print(f"  إجمالي الصور الموجودة: {len(all_images)}")
if all_images:
    print(f"  مثال: {all_images[0]}")

🔍 دور على الصور في TACO:

  عدد الصور في الـ JSON: 1500
  عدد الـ Annotations: 4784

  أمثلة على file_name:
  - batch_1/000006.jpg
  - batch_1/000008.jpg
  - batch_1/000010.jpg
  - batch_1/000019.jpg
  - batch_1/000026.jpg

  جاري البحث عن الصور...
  إجمالي الصور الموجودة: 1500
  مثال: ../datasets\TACO\TACO\data\batch_1\000000.jpg


In [14]:
# Cell 14 — Fix TACO Processing
print("🔄 معالجة TACO بشكل صح:\n")

dataset_dir = os.path.join(dataset_path, "TACO", "TACO", "data")
ann_file = os.path.join(dataset_dir, "annotations.json")

with open(ann_file, 'r') as f:
    data = json.load(f)

categories = {cat['id']: cat['name'] for cat in data['categories']}

# جمع الـ annotations لكل صورة
img_annotations = defaultdict(list)
for ann in data['annotations']:
    class_name = categories.get(ann['category_id'], '')
    if class_name in REMOVE_CLASSES:
        continue
    elif class_name in CLASS_MAPPING:
        new_class_id = FINAL_CLASSES.index(CLASS_MAPPING[class_name])
        img_annotations[ann['image_id']].append({
            'class_id': new_class_id,
            'bbox': ann['bbox']
        })

images_info = {img['id']: img for img in data['images']}

copied = 0
skipped = 0

for img_id, anns in img_annotations.items():
    if img_id not in images_info:
        skipped += 1
        continue

    img_info = images_info[img_id]
    img_w = img_info['width']
    img_h = img_info['height']
    img_file_name = img_info['file_name']  # batch_1/000006.jpg

    # المسار الكامل للصورة
    img_path = os.path.join(dataset_dir, img_file_name)

    if not os.path.exists(img_path):
        skipped += 1
        continue

    # حول الـ bbox من COCO لـ YOLO
    yolo_lines = []
    for ann in anns:
        x, y, w, h = ann['bbox']
        x_center = (x + w / 2) / img_w
        y_center = (y + h / 2) / img_h
        w_norm = w / img_w
        h_norm = h / img_h
        yolo_lines.append(
            f"{ann['class_id']} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n"
        )

    if not yolo_lines:
        skipped += 1
        continue

    # حدد الـ split
    r = random.random()
    if r < 0.7:
        split = 'train'
    elif r < 0.85:
        split = 'val'
    else:
        split = 'test'

    img_stem = Path(img_path).stem
    img_ext = Path(img_path).suffix
    new_name = f"TACO_{img_stem}"

    # انسخ الصورة
    dst_img = os.path.join(output_path, split, 'images', f"{new_name}{img_ext}")
    shutil.copy2(img_path, dst_img)

    # اكتب الـ label
    dst_label = os.path.join(output_path, split, 'labels', f"{new_name}.txt")
    with open(dst_label, 'w') as f:
        f.writelines(yolo_lines)

    copied += 1

print(f"✅ TACO: {copied} صورة اتنسخت، {skipped} اتسكبت")

🔄 معالجة TACO بشكل صح:

✅ TACO: 1481 صورة اتنسخت، 0 اتسكبت


In [15]:
# Cell 15 — Remove Old TACO Files & Verify
print("🗑️ مسح الـ TACO القديمة:\n")

deleted = 0
for split in ['train', 'val', 'test']:
    for folder in ['images', 'labels']:
        folder_path = os.path.join(output_path, split, folder)
        for f in glob.glob(f"{folder_path}/TACO_*"):
            # الملفات القديمة كانت بتاخد اسم الصورة مباشرة مش batch
            # الجديدة شكلها TACO_000006 والقديمة TACO_000006 برضو
            # هنشيل الـ duplicates بس
            pass

# بدل ده، هنعمل count للـ TACO files الحالية
print("📊 الـ TACO files الحالية:\n")
for split in ['train', 'val', 'test']:
    taco_imgs = glob.glob(f"{output_path}/{split}/images/TACO_*")
    taco_lbls = glob.glob(f"{output_path}/{split}/labels/TACO_*")
    print(f"  {split}: {len(taco_imgs)} صورة, {len(taco_lbls)} label")

🗑️ مسح الـ TACO القديمة:

📊 الـ TACO files الحالية:

  train: 238 صورة, 238 label
  val: 166 صورة, 166 label
  test: 164 صورة, 164 label


In [16]:
# Cell 16 — Fix TACO Naming (include batch name)
print("🔄 إعادة معالجة TACO مع اسم الـ batch:\n")

# امسح الـ TACO القديمة الأول
deleted = 0
for split in ['train', 'val', 'test']:
    for folder in ['images', 'labels']:
        folder_path = os.path.join(output_path, split, folder)
        for f in glob.glob(f"{folder_path}/TACO_*"):
            os.remove(f)
            deleted += 1
print(f"  🗑️ اتمسح {deleted} ملف قديم")

# إعادة المعالجة مع اسم الـ batch
dataset_dir = os.path.join(dataset_path, "TACO", "TACO", "data")
ann_file = os.path.join(dataset_dir, "annotations.json")

with open(ann_file, 'r') as f:
    data = json.load(f)

categories = {cat['id']: cat['name'] for cat in data['categories']}

img_annotations = defaultdict(list)
for ann in data['annotations']:
    class_name = categories.get(ann['category_id'], '')
    if class_name in REMOVE_CLASSES:
        continue
    elif class_name in CLASS_MAPPING:
        new_class_id = FINAL_CLASSES.index(CLASS_MAPPING[class_name])
        img_annotations[ann['image_id']].append({
            'class_id': new_class_id,
            'bbox': ann['bbox']
        })

images_info = {img['id']: img for img in data['images']}

copied = 0
skipped = 0

for img_id, anns in img_annotations.items():
    if img_id not in images_info:
        skipped += 1
        continue

    img_info = images_info[img_id]
    img_w = img_info['width']
    img_h = img_info['height']
    img_file_name = img_info['file_name']  # batch_1/000006.jpg

    img_path = os.path.join(dataset_dir, img_file_name)
    if not os.path.exists(img_path):
        skipped += 1
        continue

    yolo_lines = []
    for ann in anns:
        x, y, w, h = ann['bbox']
        x_center = (x + w / 2) / img_w
        y_center = (y + h / 2) / img_h
        w_norm = w / img_w
        h_norm = h / img_h
        yolo_lines.append(
            f"{ann['class_id']} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n"
        )

    if not yolo_lines:
        skipped += 1
        continue

    r = random.random()
    if r < 0.7:
        split = 'train'
    elif r < 0.85:
        split = 'val'
    else:
        split = 'test'

    # اسم جديد بيشمل الـ batch
    batch_name = Path(img_file_name).parent.name  # batch_1
    img_stem = Path(img_path).stem                # 000006
    img_ext = Path(img_path).suffix
    new_name = f"TACO_{batch_name}_{img_stem}"    # TACO_batch_1_000006

    dst_img = os.path.join(output_path, split, 'images', f"{new_name}{img_ext}")
    shutil.copy2(img_path, dst_img)

    dst_label = os.path.join(output_path, split, 'labels', f"{new_name}.txt")
    with open(dst_label, 'w') as f:
        f.writelines(yolo_lines)

    copied += 1

print(f"✅ TACO: {copied} صورة اتنسخت، {skipped} اتسكبت")

🔄 إعادة معالجة TACO مع اسم الـ batch:

  🗑️ اتمسح 1136 ملف قديم
✅ TACO: 1481 صورة اتنسخت، 0 اتسكبت


In [17]:
# Cell 17 — Final Verification
print("📊 الإحصائيات النهائية:\n")

total_images = 0
total_labels = 0

for split in ['train', 'val', 'test']:
    images = glob.glob(f"{output_path}/{split}/images/*")
    labels = glob.glob(f"{output_path}/{split}/labels/*")
    total_images += len(images)
    total_labels += len(labels)
    print(f"  {split}:")
    print(f"    صور:    {len(images)}")
    print(f"    labels: {len(labels)}")

print(f"\n  📊 الإجمالي:")
print(f"    صور:    {total_images}")
print(f"    labels: {total_labels}")

print("\n📊 توزيع الـ Classes:\n")
class_counts = defaultdict(int)

for label_file in glob.glob(f"{output_path}/**/*.txt", recursive=True):
    with open(label_file, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            class_id = int(line.split()[0])
            class_counts[FINAL_CLASSES[class_id]] += 1

for cls in FINAL_CLASSES:
    count = class_counts[cls]
    bar = "█" * (count // 5000)
    print(f"  {cls:<20} {count:>8} {bar}")

print(f"\n  الإجمالي: {sum(class_counts.values())}")

📊 الإحصائيات النهائية:

  train:
    صور:    25020
    labels: 25020
  val:
    صور:    6572
    labels: 6572
  test:
    صور:    6475
    labels: 6475

  📊 الإجمالي:
    صور:    38067
    labels: 38067

📊 توزيع الـ Classes:

  Plastic                108321 █████████████████████
  Glass                   11203 ██
  Metal                    9695 █
  Paper                   12292 ██
  Organic                  6130 █
  Non-Recyclable            705 

  الإجمالي: 148346
